# Assignment 4: Evaluating Adversial Learning

### Installations & Loading Libraries

In [1]:
#!pip install -q 'aif360[AdversarialDebiasing]'

In [2]:
#!pip install -q pandas numpy scikit-learn matplotlib seaborn

In [3]:
# Load all necessary packages
import sys
sys.path.append("../")
from aif360.datasets import BinaryLabelDataset
from aif360.datasets import AdultDataset, GermanDataset, CompasDataset
from aif360.metrics import BinaryLabelDatasetMetric
from aif360.metrics import ClassificationMetric
from aif360.metrics.utils import compute_boolean_conditioning_vector

from aif360.algorithms.preprocessing.optim_preproc_helpers.data_preproc_functions import load_preproc_data_adult, load_preproc_data_compas, load_preproc_data_german

from aif360.algorithms.inprocessing.adversarial_debiasing import AdversarialDebiasing

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, MaxAbsScaler
from sklearn.metrics import accuracy_score

from IPython.display import Markdown, display
import matplotlib.pyplot as plt

import tensorflow.compat.v1 as tf
tf.disable_eager_execution()

pip install 'aif360[Reductions]'
pip install 'aif360[Reductions]'
pip install 'aif360[inFairness]'
pip install 'aif360[Reductions]'


In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import random
# For reproducibility
seed = 33
np.random.seed(seed)
random.seed(seed)
tf.set_random_seed(seed)

In [5]:
# Downlaoding the raw files from datasets:
import os
import urllib.request
import aif360

# Adult dataset:
data_dir = os.path.join(os.path.dirname(aif360.__file__), "data", "raw", "adult")
os.makedirs(data_dir, exist_ok=True)
base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/"
for f in ["adult.data", "adult.test", "adult.names"]:
    urllib.request.urlretrieve(base_url + f, os.path.join(data_dir, f))

#German dataset:
data_dir = os.path.join(os.path.dirname(aif360.__file__), "data", "raw", "german")
os.makedirs(data_dir, exist_ok=True)
base_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/"
for f in ["german.data", "german.doc"]:
    urllib.request.urlretrieve(base_url + f, os.path.join(data_dir, f))

#COMPAS dataset:
data_dir = os.path.join(os.path.dirname(aif360.__file__), "data", "raw", "compas")
os.makedirs(data_dir, exist_ok=True)
base_url = "https://raw.githubusercontent.com/propublica/compas-analysis/master/"
for f in ["compas-scores-two-years.csv"]:
    urllib.request.urlretrieve(base_url + f, os.path.join(data_dir, f))

## Adult Dataset

In [6]:
# Get the dataset and split into train and test

from sklearn.model_selection import train_test_split

# The github page has a preprocessing function for the dataset already. There are two protected attributes to select from, we choose Sex.
dataset_orig = load_preproc_data_adult(protected_attributes=['sex'])
privileged_groups = [{'sex': 1}]
unprivileged_groups = [{'sex': 0}]

#Using startified train/test splits of 80% train and 20% test
train_idx, test_idx = train_test_split(
    np.arange(len(dataset_orig.labels)),
    test_size=0.2,
    random_state=seed,
    stratify=dataset_orig.labels.ravel()
)

dataset_orig_train = dataset_orig.subset(train_idx)
dataset_orig_test = dataset_orig.subset(test_idx)

/usr/local/lib/python3.13/dist-packages/aif360/algorithms/preprocessing/optim_preproc_helpers/data_preproc_functions.py:50: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sex'] = df['sex'].replace({'Female': 0.0, 'Male': 1.0})


In [7]:
# Metrics for the original dataset

# AIF360 has all relevant metrics under binary_label_dataset_metric.py
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("DP_diff= %f" % metric_orig_train.mean_difference())
print("DIR= %f" % metric_orig_train.disparate_impact())
print("Consistency= %f" % metric_orig_train.consistency().item())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original test dataset"))
print("DP_Diff= %f" % metric_orig_test.mean_difference())
print("DIR= %f" % metric_orig_test.disparate_impact())
print("Consistency= %f" % metric_orig_test.consistency().item())

#### Original training dataset

DP_diff= -0.194909
DIR= 0.358269
Consistency= 0.732337


#### Original test dataset

DP_Diff= -0.192964
DIR= 0.365123
Consistency= 0.733586


In [8]:
# Apply Adversial learning
# Learn parameters with debias set to True
tf.reset_default_graph()
sess = tf.Session()
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='debiased_classifier',
                          sess=sess,
                          seed=seed,
                          adversary_loss_weight=0.1,
                          num_epochs=30,
                          batch_size=128,
                          classifier_num_hidden_units=30,
                          debias=True)
debiased_model.fit(dataset_orig_train)
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)
sess.close()


Instructions for updating:
Please use `rate` instead of `keep_prob`. Rate should be set to `rate = 1 - keep_prob`.


epoch 0; iter: 0; batch classifier loss: 0.759462; batch adversarial loss: 0.646266
epoch 0; iter: 200; batch classifier loss: 0.583093; batch adversarial loss: 0.622361
epoch 1; iter: 0; batch classifier loss: 0.453122; batch adversarial loss: 0.628471
epoch 1; iter: 200; batch classifier loss: 0.403648; batch adversarial loss: 0.625798
epoch 2; iter: 0; batch classifier loss: 0.478908; batch adversarial loss: 0.656462
epoch 2; iter: 200; batch classifier loss: 0.488861; batch adversarial loss: 0.632111
epoch 3; iter: 0; batch classifier loss: 0.447319; batch adversarial loss: 0.651430
epoch 3; iter: 200; batch classifier loss: 0.437893; batch adversarial loss: 0.567125
epoch 4; iter: 0; batch classifier loss: 0.462277; batch adversarial loss: 0.624380
epoch 4; iter: 200; batch classifier loss: 0.454944; batch adversarial loss: 0.616684
epoch 5; iter: 0; batch classifier loss: 0.409194; batch adversarial loss: 0.592563
epoch 5; iter: 200; batch classifier loss: 0.473983; batch adversa

In [9]:
# Calculating fairness metrics after bias mitigation
metric_debiased_train = BinaryLabelDatasetMetric(dataset_debiasing_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased training dataset"))
print("DP_diff= %f" % metric_debiased_train.mean_difference())
print("DIR= %f" % metric_debiased_train.disparate_impact())
print("Consistency= %f" % metric_debiased_train.consistency().item())
metric_debiased_test = BinaryLabelDatasetMetric(dataset_debiasing_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased test dataset"))
print("DP_Diff= %f" % metric_debiased_test.mean_difference())
print("DIR= %f" % metric_debiased_test.disparate_impact())
print("Consistency= %f" % metric_debiased_test.consistency().item())


#### Debiased training dataset

DP_diff= -0.065790
DIR= 0.680551
Consistency= 0.999969


#### Debiased test dataset

DP_Diff= -0.063094
DIR= 0.683164
Consistency= 0.999980


In [10]:
# Accuracy used as the predictive-performance measure
# Pre-debiasing accuracy (original labels as predictions)
accuracy_original = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_orig_test.labels.ravel())
print("Original test accuracy: %.4f" % accuracy_original)
# Post-debiasing accuracy
accuracy_debiasing = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_debiasing_test.labels.ravel()
)

print("Debiased test accuracy: %.4f" % accuracy_debiasing)

Original test accuracy: 1.0000
Debiased test accuracy: 0.7868


## German Dataset

In [11]:
# Get the dataset and split into train and test

# There are again two protected attributes to select from, we choose Sex.
dataset_orig = load_preproc_data_german(protected_attributes=['sex'])
privileged_groups = [{'sex': 1}] #Male
unprivileged_groups = [{'sex': 0}] #Female

#Using startified train/test splits of 80% train and 20% test
train_idx, test_idx = train_test_split(
    np.arange(len(dataset_orig.labels)),
    test_size=0.2,
    random_state=seed,
    stratify=dataset_orig.labels.ravel()
)

dataset_orig_train = dataset_orig.subset(train_idx)
dataset_orig_test = dataset_orig.subset(test_idx)


/usr/local/lib/python3.13/dist-packages/aif360/algorithms/preprocessing/optim_preproc_helpers/data_preproc_functions.py:261: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sex'] = df['personal_status'].replace(status_map)


In [12]:
# Metrics for the original dataset

# AIF360 has all relevant metrics under binary_label_dataset_metric.py
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("DP_diff= %f" % metric_orig_train.mean_difference())
print("DIR= %f" % metric_orig_train.disparate_impact())
print("Consistency= %f" % metric_orig_train.consistency().item())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original test dataset"))
print("DP_Diff= %f" % metric_orig_test.mean_difference())
print("DIR= %f" % metric_orig_test.disparate_impact())
print("Consistency= %f" % metric_orig_test.consistency().item())

#### Original training dataset

DP_diff= -0.091168
DIR= 0.875049
Consistency= 0.642000


#### Original test dataset

DP_Diff= 0.000000
DIR= 1.000000
Consistency= 0.660000


In [13]:
# Apply Adversial learning
# Learn parameters with debias set to True
tf.reset_default_graph()
sess = tf.Session()
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='german_debiased_classifier',
                          sess=sess,
                          seed=seed,
                          adversary_loss_weight=0.1,
                          num_epochs=30,
                          batch_size=128,
                          classifier_num_hidden_units=30,
                          debias=True)
debiased_model.fit(dataset_orig_train)
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)
sess.close()


epoch 0; iter: 0; batch classifier loss: 0.706176; batch adversarial loss: 0.633838
epoch 1; iter: 0; batch classifier loss: 0.650517; batch adversarial loss: 0.614940
epoch 2; iter: 0; batch classifier loss: 0.647856; batch adversarial loss: 0.568565
epoch 3; iter: 0; batch classifier loss: 0.637633; batch adversarial loss: 0.631072
epoch 4; iter: 0; batch classifier loss: 0.663206; batch adversarial loss: 0.625527
epoch 5; iter: 0; batch classifier loss: 0.597611; batch adversarial loss: 0.663370
epoch 6; iter: 0; batch classifier loss: 0.630000; batch adversarial loss: 0.593524
epoch 7; iter: 0; batch classifier loss: 0.605414; batch adversarial loss: 0.634067
epoch 8; iter: 0; batch classifier loss: 0.584244; batch adversarial loss: 0.618418
epoch 9; iter: 0; batch classifier loss: 0.589364; batch adversarial loss: 0.654421
epoch 10; iter: 0; batch classifier loss: 0.603546; batch adversarial loss: 0.623785
epoch 11; iter: 0; batch classifier loss: 0.598026; batch adversarial loss:

In [14]:
# Calculating fairness metrics after bias mitigation
metric_debiased_train = BinaryLabelDatasetMetric(dataset_debiasing_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased training dataset"))
print("DP_diff= %f" % metric_debiased_train.mean_difference())
print("DIR= %f" % metric_debiased_train.disparate_impact())
print("Consistency= %f" % metric_debiased_train.consistency().item())
metric_debiased_test = BinaryLabelDatasetMetric(dataset_debiasing_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased test dataset"))
print("DP_Diff= %f" % metric_debiased_test.mean_difference())
print("DIR= %f" % metric_debiased_test.disparate_impact())
print("Consistency= %f" % metric_debiased_test.consistency().item())


#### Debiased training dataset

DP_diff= 0.000000
DIR= 1.000000
Consistency= 1.000000


#### Debiased test dataset

DP_Diff= 0.000000
DIR= 1.000000
Consistency= 1.000000


In [15]:
# Accuracy used as the predictive-performance measure
# Pre-debiasing accuracy (original labels as predictions)
accuracy_original = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_orig_test.labels.ravel())
print("Original test accuracy: %.4f" % accuracy_original)
# Post-debiasing accuracy
accuracy_debiasing = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_debiasing_test.labels.ravel()
)

print("Debiased test accuracy: %.4f" % accuracy_debiasing)

Original test accuracy: 1.0000
Debiased test accuracy: 0.7000


## COMPAS Dataset

In [16]:
# Get the dataset and split into train and test

# This time the protected attributes is Race.
dataset_orig = load_preproc_data_compas(protected_attributes=['race'])
privileged_groups = [{'race': 1}] #White
unprivileged_groups = [{'race': 0}]

#Using startified train/test splits of 80% train and 20% test
train_idx, test_idx = train_test_split(
    np.arange(len(dataset_orig.labels)),
    test_size=0.2,
    random_state=seed,
    stratify=dataset_orig.labels.ravel()
)

dataset_orig_train = dataset_orig.subset(train_idx)
dataset_orig_test = dataset_orig.subset(test_idx)


/usr/local/lib/python3.13/dist-packages/aif360/algorithms/preprocessing/optim_preproc_helpers/data_preproc_functions.py:163: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dfcutQ['sex'] = dfcutQ['sex'].replace({'Female': 1.0, 'Male': 0.0})


In [17]:
# Metrics for the original dataset

# AIF360 has all relevant metrics under binary_label_dataset_metric.py
metric_orig_train = BinaryLabelDatasetMetric(dataset_orig_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original training dataset"))
print("DP_diff= %f" % metric_orig_train.mean_difference())
print("DIR= %f" % metric_orig_train.disparate_impact())
print("Consistency= %f" % metric_orig_train.consistency().item())
metric_orig_test = BinaryLabelDatasetMetric(dataset_orig_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Original test dataset"))
print("DP_Diff= %f" % metric_orig_test.mean_difference())
print("DIR= %f" % metric_orig_test.disparate_impact())
print("Consistency= %f" % metric_orig_test.consistency().item())

#### Original training dataset

DP_diff= -0.135395
DIR= 0.778590
Consistency= 0.589105


#### Original test dataset

DP_Diff= -0.120129
DIR= 0.799785
Consistency= 0.588826


In [18]:
# Apply Adversial learning
# Learn parameters with debias set to True
tf.reset_default_graph()
sess = tf.Session()
debiased_model = AdversarialDebiasing(privileged_groups = privileged_groups,
                          unprivileged_groups = unprivileged_groups,
                          scope_name='compas_debiased_classifier',
                          sess=sess,
                          seed=seed,
                          adversary_loss_weight=0.1,
                          num_epochs=30,
                          batch_size=128,
                          classifier_num_hidden_units=30,
                          debias=True)
debiased_model.fit(dataset_orig_train)
dataset_debiasing_train = debiased_model.predict(dataset_orig_train)
dataset_debiasing_test = debiased_model.predict(dataset_orig_test)
sess.close()


epoch 0; iter: 0; batch classifier loss: 0.695051; batch adversarial loss: 0.737304
epoch 1; iter: 0; batch classifier loss: 0.722856; batch adversarial loss: 0.774049
epoch 2; iter: 0; batch classifier loss: 0.725623; batch adversarial loss: 0.808136
epoch 3; iter: 0; batch classifier loss: 0.646721; batch adversarial loss: 0.750119
epoch 4; iter: 0; batch classifier loss: 0.640356; batch adversarial loss: 0.751634
epoch 5; iter: 0; batch classifier loss: 0.599841; batch adversarial loss: 0.737207
epoch 6; iter: 0; batch classifier loss: 0.637243; batch adversarial loss: 0.744973
epoch 7; iter: 0; batch classifier loss: 0.609934; batch adversarial loss: 0.723220
epoch 8; iter: 0; batch classifier loss: 0.619959; batch adversarial loss: 0.724733
epoch 9; iter: 0; batch classifier loss: 0.671567; batch adversarial loss: 0.718265
epoch 10; iter: 0; batch classifier loss: 0.657360; batch adversarial loss: 0.706227
epoch 11; iter: 0; batch classifier loss: 0.644095; batch adversarial loss:

In [19]:
# Calculating fairness metrics after bias mitigation
metric_debiased_train = BinaryLabelDatasetMetric(dataset_debiasing_train,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased training dataset"))
print("DP_diff= %f" % metric_debiased_train.mean_difference())
print("DIR= %f" % metric_debiased_train.disparate_impact())
print("Consistency= %f" % metric_debiased_train.consistency().item())
metric_debiased_test = BinaryLabelDatasetMetric(dataset_debiasing_test,
                                             unprivileged_groups=unprivileged_groups,
                                             privileged_groups=privileged_groups)
display(Markdown("#### Debiased test dataset"))
print("DP_Diff= %f" % metric_debiased_test.mean_difference())
print("DIR= %f" % metric_debiased_test.disparate_impact())
print("Consistency= %f" % metric_debiased_test.consistency().item())


#### Debiased training dataset

DP_diff= -0.028613
DIR= 0.954329
Consistency= 1.000000


#### Debiased test dataset

DP_Diff= -0.092942
DIR= 0.859129
Consistency= 0.993750


In [20]:
# Accuracy used as the predictive-performance measure
# Pre-debiasing accuracy (original labels as predictions)
accuracy_original = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_orig_test.labels.ravel())
print("Original test accuracy: %.4f" % accuracy_original)
# Post-debiasing accuracy
accuracy_debiasing = accuracy_score(
    dataset_orig_test.labels.ravel(),
    dataset_debiasing_test.labels.ravel()
)

print("Debiased test accuracy: %.4f" % accuracy_debiasing)

Original test accuracy: 1.0000
Debiased test accuracy: 0.6420
